# V7 核心算法测试 Notebook

本 notebook 测试 V7 市场状态量化系统的核心算法：

| 模块 | 核心能力 | 测试内容 |
|------|----------|----------|
| MarketRegimeEngine | 体制识别 | 波动率/动量/广度 → softmax概率 |
| MarketStateClassifier | 3D分类 | 估值/动量/体制 → 九宫格状态 |
| DerivativesSignalEngine | 衍生品信号 | 商品信号/期限结构/基差 |
| ContractManager | 动态合约 | 近月/远月/期权推导 |
| 风险评估 | 综合风险 | 多维度风险因子评估 |


In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

import yaml
config_path = PROJECT_ROOT / 'config' / 'market_state' / 'system_config.yaml'
# config_path = PROJECT_ROOT / 'config' / 'market_state_system' / 'system_config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print(f'配置加载完成 | V{config.get("system", {}).get("version")}')

配置加载完成 | V7.0.0


In [2]:
# 构建完整 V7 服务栈
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from data_services.tdx_adapter import TDXAdapter
from data_services.database_reader import DatabaseReader
from data_services.ak_adapter import AKAdapter
from data_services.data_loading_service import DataLoadingService
from market_state_system.core.contract_manager import ContractManager

# 配置服务
cfg_svc = ConfigService(system_name='market_state_system', enable_hot_reload=False)

# TDX 适配器
tdx = TDXAdapter({
    'host': config.get('tdx', {}).get('hq_host', '180.153.18.170'),
    'port': config.get('tdx', {}).get('hq_port', 7709),
    'pool_size': 3,
    'retry_count': 2,
})

# 数据库
from config.global_settings import DATABASE_ENGINES, DB_POOL_CONFIG
db = DatabaseReader(DATABASE_ENGINES, DB_POOL_CONFIG)

# AKShare
ak = AKAdapter()

# 数据服务
data_svc = DataLoadingService(
    config_service=cfg_svc,
    database_reader=db,
    tdx_adapter=tdx,
    ak_adapter=ak,
    cache_service=CacheService(max_size=200, ttl=3600),
)

print(f'服务栈初始化完成 | TDX={"可用" if tdx.is_available() else "不可用"}')

服务栈初始化完成 | TDX=可用


## 1. MarketRegimeEngine — 市场体制识别

四体制识别: Bull(牛市) / Bear(熊市) / Volatile(震荡) / Recovery(复苏)
三维信号: 波动率 + 动量 + 市场广度 → softmax → 滞后确认

In [3]:
from market_state_system.core.market_regime_engine import MarketRegimeEngine

regime_engine = MarketRegimeEngine(data_svc, config)
print(f'MarketRegimeEngine 初始化完成')
print(f'  确认天数: {regime_engine.confirmation_days}')
print(f'  体制配置: {list(regime_engine.regime_config.keys())}')

MarketRegimeEngine 初始化完成
  确认天数: 3
  体制配置: ['bull', 'bear', 'volatile', 'recovery']


In [4]:
# 执行体制检测
regime_result = regime_engine.detect_regime()

print('=== 市场体制检测结果 ===')
print(f'当前体制: {regime_result["current_regime"]}')
print(f'置信度: {regime_result["confidence"]:.2%}')
print(f'确认天数: {regime_result["confirmation_days"]}')
print(f'\n波动率: 20d={regime_result["volatility_20d"]:.4f} | 60d={regime_result["volatility_60d"]:.4f}')
print(f'动量:   20d={regime_result["momentum_20d"]:.4f} | 60d={regime_result["momentum_60d"]:.4f}')
print(f'\n体制概率分布:')
for regime, prob in regime_result['regime_probability'].items():
    bar = '█' * int(prob * 50)
    label = {'bull': '牛市', 'bear': '熊市', 'volatile': '震荡', 'recovery': '复苏'}.get(regime, regime)
    print(f'  {label}({regime:>8s}): {prob:.2%} {bar}')

⚠️ 未找到 market_benchmarks 配置，广度分数默认 0.5


=== 市场体制检测结果 ===
当前体制: recovery
置信度: 36.21%
确认天数: 1

波动率: 20d=0.1514 | 60d=0.1728
动量:   20d=0.0219 | 60d=0.0419

体制概率分布:
  牛市(    bull): 18.65% █████████
  熊市(    bear): 21.32% ██████████
  震荡(volatile): 23.82% ███████████
  复苏(recovery): 36.21% ██████████████████


### 1.1 波动率计算单元测试

验证 20d/60d 年化滚动波动率的正确性

In [5]:
# 波动率计算单元测试
df_benchmark = regime_engine._load_benchmark_data()
if df_benchmark is not None and len(df_benchmark) >= 60:
    vol_20d, vol_60d = regime_engine._calc_volatility(df_benchmark)
    
    # 手动验证
    returns = df_benchmark['close'].pct_change().dropna()
    manual_vol_20 = returns.tail(20).std() * np.sqrt(252)
    manual_vol_60 = returns.tail(60).std() * np.sqrt(252)
    
    print('波动率计算验证:')
    print(f'  引擎计算: vol_20d={vol_20d:.6f}, vol_60d={vol_60d:.6f}')
    print(f'  手动计算: vol_20d={manual_vol_20:.6f}, vol_60d={manual_vol_60:.6f}')
    print(f'  差异:     20d={abs(vol_20d-manual_vol_20):.8f}, 60d={abs(vol_60d-manual_vol_60):.8f}')
    print(f'  ✅ 一致性检验: {"通过" if abs(vol_20d-manual_vol_20)<1e-6 and abs(vol_60d-manual_vol_60)<1e-6 else "失败"}')
else:
    print('⚠️ 基准数据不足')

波动率计算验证:
  引擎计算: vol_20d=0.151437, vol_60d=0.172797
  手动计算: vol_20d=0.151437, vol_60d=0.172797
  差异:     20d=0.00000000, 60d=0.00000000
  ✅ 一致性检验: 通过


### 1.2 Softmax 概率转换单元测试

In [6]:
# Softmax 概率转换单元测试
test_scores = {
    'bull': 0.6,
    'bear': 0.15,
    'volatile': 0.35,
    'recovery': 0.25,
}

probs = MarketRegimeEngine._softmax_probabilities(test_scores)
print('Softmax 概率转换测试:')
print(f'  输入得分: {test_scores}')
print(f'  输出概率: {probs}')
print(f'  概率总和: {sum(probs.values()):.6f}')
print(f'  ✅ 概率和为1: {"通过" if abs(sum(probs.values())-1.0)<1e-6 else "失败"}')

# 测试极端情况
extreme_scores = {'bull': 10.0, 'bear': 0.01, 'volatile': 0.01, 'recovery': 0.01}
extreme_probs = MarketRegimeEngine._softmax_probabilities(extreme_scores)
print(f'\n极端得分测试: {extreme_probs}')
print(f'  牛市概率占比: {extreme_probs["bull"]:.2%} (预期接近100%)')

Softmax 概率转换测试:
  输入得分: {'bull': 0.6, 'bear': 0.15, 'volatile': 0.35, 'recovery': 0.25}
  输出概率: {'bull': 0.3204, 'bear': 0.2043, 'volatile': 0.2495, 'recovery': 0.2258}
  概率总和: 1.000000
  ✅ 概率和为1: 通过

极端得分测试: {'bull': 0.9999, 'bear': 0.0, 'volatile': 0.0, 'recovery': 0.0}
  牛市概率占比: 99.99% (预期接近100%)


## 2. MarketStateClassifier — 3D 市场状态分类

三维模型: 估值(D1) + 动量(D2) + 体制(D3) → 综合得分 → 九宫格状态

In [7]:
from market_state_system.core.market_state_classifier import MarketStateClassifier, GRID_STATES

classifier = MarketStateClassifier(data_svc, config, regime_engine)
print(f'MarketStateClassifier 初始化完成')
print(f'  权重: 估值={classifier.w_valuation:.0%} 动量={classifier.w_momentum:.0%} 体制={classifier.w_regime:.0%}')
print(f'  九宫格: {GRID_STATES}')

MarketStateClassifier 初始化完成
  权重: 估值=35% 动量=35% 体制=30%
  九宫格: ['战略进攻区', '积极配置区', '均衡持有区', '防御观望区', '战略防御区', '防御进攻区', '左侧布局区', '左侧防御区', '谨慎持有区']


In [8]:
# 执行3D分类
classification_result = classifier.classify()

print('=== 3D 市场状态分类结果 ===')
print(f'市场状态: {classification_result["market_state"]}')
print(f'综合得分: {classification_result["composite_score"]:.2f}')
print(f'置信度:   {classification_result["confidence"]:.2%}')
print(f'体制名称: {classification_result["regime_name"]}')
print(f'\n三维得分:')
print(f'  估值得分: {classification_result["valuation_score"]:.2f}')
print(f'  动量得分: {classification_result["momentum_score"]:.2f}')
print(f'  体制得分: {classification_result["regime_score"]:.2f}')
print(f'\n诊断详情:')
for dim, diag in classification_result['diagnosis'].items():
    print(f'  [{dim}] score={diag.get("score", "N/A")} | {diag}')

⚠️ 未找到 market_benchmarks 配置，广度分数默认 0.5


=== 3D 市场状态分类结果 ===
市场状态: 均衡持有区
综合得分: 51.35
置信度:   53.14%
体制名称: 复苏期

三维得分:
  估值得分: 10.47
  动量得分: 90.77
  体制得分: 53.06

诊断详情:
  [valuation] score=10.5 | {'score': 10.5, 'level': '极度高估', 'indices': {'000300': {'name': '沪深300', 'pe_ttm': 14.6, 'percentile': 79.0, 'score': 21.0}, '000905': {'name': '中证500', 'pe_ttm': 29.15, 'percentile': 96.9, 'score': 3.1}, '000852': {'name': '中证1000', 'pe_ttm': 37.92, 'percentile': 99.7, 'score': 0.3}}}
  [momentum] score=90.8 | {'score': np.float64(90.8), 'trend': '强上升趋势', 'indices': {'000300': {'name': '大盘', 'above_ma20': True, 'above_ma60': True, 'golden_cross': True, 'deviation_20': 0.0103, 'deviation_60': 0.044, 'score': np.float64(88.4)}, '000905': {'name': '中盘', 'above_ma20': True, 'above_ma60': True, 'golden_cross': True, 'deviation_20': 0.0141, 'deviation_60': 0.0541, 'score': np.float64(92.1)}, '000852': {'name': '小盘', 'above_ma20': True, 'above_ma60': True, 'golden_cross': True, 'deviation_20': 0.0171, 'deviation_60': 0.0639, 'score': np.float6

### 2.1 估值维度深入分析

PE百分位 → 估值得分映射: 低百分位=高分(低估), 高百分位=低分(高估)

In [9]:
# 估值维度深入分析
val_score, val_diag = classifier._calc_valuation_score()

print('=== 估值维度分析 ===')
print(f'估值得分: {val_score:.2f}')
print(f'估值水平: {val_diag["level"]}')
print(f'\n各指数PE百分位:')
for code, info in val_diag.get('indices', {}).items():
    if isinstance(info, dict) and info.get('pe_ttm') is not None:
        bar_len = int(info['percentile'] / 2)
        bar = '█' * bar_len + '░' * (50 - bar_len)
        print(f'  {info["name"]}({code}): PE={info["pe_ttm"]:.2f} 百分位={info["percentile"]:.1f}% {bar} 得分={info["score"]:.1f}')
    else:
        print(f'  {code}: 数据不可用 {info}')

=== 估值维度分析 ===
估值得分: 10.47
估值水平: 极度高估

各指数PE百分位:
  沪深300(000300): PE=14.60 百分位=79.0% ███████████████████████████████████████░░░░░░░░░░░ 得分=21.0
  中证500(000905): PE=29.15 百分位=96.9% ████████████████████████████████████████████████░░ 得分=3.1
  中证1000(000852): PE=37.92 百分位=99.7% █████████████████████████████████████████████████░ 得分=0.3


### 2.2 动量维度深入分析

均线趋势分析: 价格vs MA20/MA60, 金叉/死叉, 趋势强度

In [10]:
# 动量维度深入分析
mom_score, mom_diag = classifier._calc_momentum_score()

print('=== 动量维度分析 ===')
print(f'动量得分: {mom_score:.2f}')
print(f'趋势方向: {mom_diag["trend"]}')
print(f'\n各指数均线分析:')
for code, info in mom_diag.get('indices', {}).items():
    if isinstance(info, dict) and info.get('score') is not None:
        ma20_status = '🟢站上' if info.get('above_ma20') else '🔴跌破'
        ma60_status = '🟢站上' if info.get('above_ma60') else '🔴跌破'
        cross = '金叉✅' if info.get('golden_cross') else '死叉❌'
        print(f'  {info["name"]}({code}): MA20={ma20_status} MA60={ma60_status} {cross} '
              f'偏离20d={info.get("deviation_20",0):.4f} 偏离60d={info.get("deviation_60",0):.4f} '
              f'得分={info["score"]}')
    else:
        print(f'  {code}: 数据不足')

=== 动量维度分析 ===
动量得分: 90.77
趋势方向: 强上升趋势

各指数均线分析:
  大盘(000300): MA20=🟢站上 MA60=🟢站上 金叉✅ 偏离20d=0.0103 偏离60d=0.0440 得分=88.4
  中盘(000905): MA20=🟢站上 MA60=🟢站上 金叉✅ 偏离20d=0.0141 偏离60d=0.0541 得分=92.1
  小盘(000852): MA20=🟢站上 MA60=🟢站上 金叉✅ 偏离20d=0.0171 偏离60d=0.0639 得分=93.5
  微盘(932000): MA20=🟢站上 MA60=🟢站上 金叉✅ 偏离20d=0.0119 偏离60d=0.0529 得分=91.0


### 2.3 九宫格映射边界测试

测试不同综合得分下的九宫格状态映射

In [11]:
# 九宫格映射边界测试
test_cases = [
    # (composite, val, mom, regime_name, expected)
    (85, 80, 85, 'bull', '战略进攻区'),
    (70, 65, 70, 'bull', '积极配置区'),
    (55, 50, 55, 'volatile', '均衡持有区'),
    (40, 35, 40, 'volatile', '防御观望区'),
    (15, 20, 15, 'bear', '战略防御区'),
    (30, 75, 35, 'recovery', '左侧布局区'),   # 估值极低+体制复苏
    (30, 75, 35, 'bear', '左侧防御区'),        # 估值极低+体制熊市
    (45, 40, 65, 'recovery', '防御进攻区'),     # 综合低但动量改善
    (50, 50, 50, 'volatile', '谨慎持有区'),     # 估值中性+体制震荡
]

print('九宫格映射边界测试:')
print(f'{"综合":>5} {"估值":>5} {"动量":>5} {"体制":>10} → {"预期":<10} {"实际":<10} {"结果"}')
print('-' * 65)

all_pass = True
for composite, val, mom, regime_name, expected in test_cases:
    regime_mock = {'current_regime': regime_name, 'regime_probability': {}}
    actual = classifier._map_to_grid(composite, val, mom, regime_mock)
    passed = actual == expected
    all_pass = all_pass and passed
    mark = '✅' if passed else '❌'
    print(f'{composite:>5} {val:>5} {mom:>5} {regime_name:>10} → {expected:<10} {actual:<10} {mark}')

print(f'\n总体结果: {"✅ 全部通过" if all_pass else "❌ 存在失败"}')

九宫格映射边界测试:
   综合    估值    动量         体制 → 预期         实际         结果
-----------------------------------------------------------------
   85    80    85       bull → 战略进攻区      战略进攻区      ✅
   70    65    70       bull → 积极配置区      积极配置区      ✅
   55    50    55   volatile → 均衡持有区      谨慎持有区      ❌
   40    35    40   volatile → 防御观望区      防御观望区      ✅
   15    20    15       bear → 战略防御区      战略防御区      ✅
   30    75    35   recovery → 左侧布局区      左侧布局区      ✅
   30    75    35       bear → 左侧防御区      左侧防御区      ✅
   45    40    65   recovery → 防御进攻区      防御进攻区      ✅
   50    50    50   volatile → 谨慎持有区      谨慎持有区      ✅

总体结果: ❌ 存在失败


## 3. DerivativesSignalEngine — 衍生品信号引擎

统一合并 V6.1 的 CommodityEngineService + FuturesAnalysisService
核心方法: commodity_signals / term_structure / index_futures_basis / industry_sentiment

In [12]:
from market_state_system.core.derivatives_signal_engine import DerivativesSignalEngine

# 初始化 ContractManager
cm = ContractManager(reference_date=datetime.now())

# 初始化衍生品信号引擎
deriv_engine = DerivativesSignalEngine(data_svc, contract_manager=cm, config=config)
print(f'DerivativesSignalEngine 初始化完成')
print(f'  商品策略数: {len(deriv_engine._smap)}')

DerivativesSignalEngine 初始化完成
  商品策略数: 8


In [13]:
# 测试商品期货信号
commodity_signals = deriv_engine.calculate_commodity_signals()
print(f'=== 商品期货信号 ({len(commodity_signals)}品种) ===')
for code, sig in commodity_signals.items():
    print(f'  {sig["name"]}({code}): 20d涨跌={sig["price_chg_20d"]:+.2f}% → {sig["signal"]} (score={sig["score"]:+.2f})')
    print(f'    影响方向: {sig["directions"]} | 近月={sig["near_contract"]} 远月={sig["far_contract"]}')

=== 商品期货信号 (8品种) ===
  沪铜(CUL8): 20d涨跌=+2.53% → 成本稳定 (score=+0.00)
    影响方向: ['高端制造', '供应链'] | 近月=CU2606 远月=CU2607
  沪铝(ALL8): 20d涨跌=-2.12% → 成本稳定 (score=+0.00)
    影响方向: ['高端制造', '新能源'] | 近月=AL2606 远月=AL2607
  碳酸锂(LCL8): 20d涨跌=+2.17% → 成本稳定 (score=+0.00)
    影响方向: ['新能源', '信息技术'] | 近月=LC2606 远月=LC2607
  工业硅(SIL8): 20d涨跌=-1.10% → 成本稳定 (score=+0.00)
    影响方向: ['信息技术', '新能源'] | 近月=SI2606 远月=SI2607
  原油(SCL8): 20d涨跌=-5.33% → 成本稳定 (score=+0.00)
    影响方向: ['公用事业', '供应链', '传统升级'] | 近月=SC2606 远月=SC2607
  螺纹钢(RBL8): 20d涨跌=+1.10% → 正常 (score=+0.00)
    影响方向: ['传统升级', '供应链'] | 近月=RB2606 远月=RB2607
  豆粕(ML8): 20d涨跌=-0.47% → 成本稳定 (score=+0.00)
    影响方向: ['现代农业', '生物健康', '文化消费'] | 近月=M2607 远月=M2608
  黄金(AUL8): 20d涨跌=-4.15% → 利空信号 (score=-0.05)
    影响方向: ['公用事业'] | 近月=AU2606 远月=AU2608


In [14]:
# 测试期限结构
term_structure = deriv_engine.calculate_term_structure()
print(f'=== 期限结构 ({len(term_structure)}品种) ===')
for key, ts in term_structure.items():
    struct_label = '📈Back(供应紧张)' if ts['structure'] == 'backwardation' else '📉Cont(供应充足)'
    print(f'  {key}: 价差={ts["spread"]:+.2f}% {struct_label} | 近月={ts["near_code"]}({ts["near_price"]:.0f}) 远月={ts["far_code"]}({ts["far_price"]:.0f})')

ValueError: too many values to unpack (expected 3, got 8)

In [24]:
# 测试股指期货基差
basis_results = deriv_engine.calculate_index_futures_basis()
print(f'=== 股指期货基差 ({len(basis_results)}品种) ===')
for key, data in basis_results.items():
    print(f'  {data["description"]}({key}): 基差={data["basis_pct"]:+.2f}% → {data["signal"]}')
    print(f'    期货={data["futures_price"]:.2f} 现货={data["spot_price"]:.2f} 差额={data["basis"]:.2f}')

ValueError: too many values to unpack (expected 3, got 8)

In [16]:
# 测试产业景气度
industry_sentiment = deriv_engine.calculate_industry_sentiment(term_structure)
print('=== 产业景气度 ===')
sorted_sent = sorted(industry_sentiment.items(), key=lambda x: x[1], reverse=True)
for direction, score in sorted_sent:
    bar_len = int(score / 2)
    bar = '█' * bar_len + '░' * (50 - bar_len)
    label = '🔥景气' if score > 65 else ('📊稳健' if score > 50 else '❄️疲软')
    print(f'  {direction}: {score:.0f}分 {bar} {label}')

NameError: name 'term_structure' is not defined

## 4. 动态阈值 — 百分位法

V7 核心改进：使用历史百分位法计算动态阈值，取代 V6 的固定阈值

In [17]:
# 动态阈值百分位法测试
def calc_dynamic_threshold(series, percentiles=[10, 25, 50, 75, 90]):
    """计算序列的动态百分位阈值"""
    return {f'p{p}': float(np.percentile(series.dropna(), p)) for p in percentiles}

# 对沪深300的PE_TTM计算动态阈值
df_pe = data_svc.load_pe_data('000300')
if not df_pe.empty and 'pe_ttm' in df_pe.columns:
    pe_series = df_pe['pe_ttm'].dropna()
    thresholds = calc_dynamic_threshold(pe_series)
    current_pe = pe_series.iloc[-1]
    current_percentile = float((pe_series < current_pe).sum() / len(pe_series) * 100)
    
    print('=== 沪深300 PE动态阈值(百分位法) ===')
    print(f'数据量: {len(pe_series)}条 | 当前PE: {current_pe:.2f} | 当前百分位: {current_percentile:.1f}%')
    print(f'\n百分位阈值:')
    for label, value in thresholds.items():
        print(f'  {label}: PE={value:.2f}')
    
    # 映射到状态
    if current_percentile < 25:
        state = '极度低估 — 战略进攻区间'
    elif current_percentile < 40:
        state = '偏低估 — 积极配置区间'
    elif current_percentile < 60:
        state = '中性 — 均衡持有区间'
    elif current_percentile < 75:
        state = '偏高估 — 防御观望区间'
    else:
        state = '极度高估 — 战略防御区间'
    print(f'\n当前状态: {state}')
else:
    print('⚠️ PE数据不可用')

=== 沪深300 PE动态阈值(百分位法) ===
数据量: 1544条 | 当前PE: 14.60 | 当前百分位: 79.0%

百分位阈值:
  p10: PE=11.28
  p25: PE=12.05
  p50: PE=12.86
  p75: PE=14.41
  p90: PE=15.56

当前状态: 极度高估 — 战略防御区间


In [18]:
# PCR动态阈值测试
pcr_thresholds = config.get('risk_thresholds', {}).get('pcr', {})
print('=== PCR 风险阈值配置 ===')
for key, value in pcr_thresholds.items():
    print(f'  {key}: {value}')

print(f'\nPCR判断逻辑:')
print(f'  PCR > {pcr_thresholds.get("extreme_high", 1.5)}: 极度恐惧(看跌过度) → 逆向看多')
print(f'  PCR > {pcr_thresholds.get("warning_high", 1.3)}: 恐惧预警')
print(f'  PCR < {pcr_thresholds.get("extreme_low", 0.5)}: 极度贪婪(看涨过度) → 逆向看空')
print(f'  PCR < {pcr_thresholds.get("warning_low", 0.7)}: 贪婪预警')
print(f'  中性区间: {pcr_thresholds.get("warning_low", 0.7)} ~ {pcr_thresholds.get("warning_high", 1.3)}')

=== PCR 风险阈值配置 ===
  warning_high: 1.3
  warning_low: 0.7
  extreme_high: 1.5
  extreme_low: 0.5
  ma_window: 5

PCR判断逻辑:
  PCR > 1.5: 极度恐惧(看跌过度) → 逆向看多
  PCR > 1.3: 恐惧预警
  PCR < 0.5: 极度贪婪(看涨过度) → 逆向看空
  PCR < 0.7: 贪婪预警
  中性区间: 0.7 ~ 1.3


## 5. 置信度分析

三维方向一致性 → 置信度：三维得分越一致，置信度越高

In [19]:
# 置信度分析测试
test_scenarios = [
    ('三维同高', np.array([85, 80, 75])),
    ('三维同低', np.array([20, 15, 25])),
    ('三维中性', np.array([50, 48, 52])),
    ('估值分化', np.array([80, 35, 45])),
    ('动量分化', np.array([40, 85, 50])),
    ('极端分化', np.array([90, 10, 50])),
]

print('=== 置信度场景测试 ===')
print(f'{"场景":<12} {"估值":>6} {"动量":>6} {"体制":>6} {"标准差":>8} {"置信度":>8}')
print('-' * 55)
for name, scores in test_scenarios:
    conf = classifier._calc_confidence(scores[0], scores[1], scores[2])
    std = float(np.std(scores))
    print(f'{name:<12} {scores[0]:>6.0f} {scores[1]:>6.0f} {scores[2]:>6.0f} {std:>8.2f} {conf:>8.2%}')

=== 置信度场景测试 ===
场景               估值     动量     体制      标准差      置信度
-------------------------------------------------------
三维同高             85     80     75     4.08  100.00%
三维同低             20     15     25     4.08  100.00%
三维中性             50     48     52     1.63   97.67%
估值分化             80     35     45    19.29   72.44%
动量分化             40     85     50    19.29   72.44%
极端分化             90     10     50    32.66   53.34%


## 6. 完整管线集成测试

运行 V7 完整分析管线: 合约管理 → 衍生品信号 → 体制检测 → 状态分类

In [20]:
# 完整管线集成测试
import time
start = time.time()

print('='*60)
print('  V7 完整管线集成测试')
print('='*60)

# Step 1: 合约推导
t1 = time.time()
commodity_pairs = cm.get_commodity_contracts()
idx_futures = cm.get_index_futures_contracts()
print(f'\n[1] 合约推导: {len(commodity_pairs)}商品 + {len(idx_futures)}股指 ({time.time()-t1:.2f}s)')

# Step 2: 衍生品信号
t2 = time.time()
deriv_report = deriv_engine.generate_report()
print(f'[2] 衍生品信号: {len(deriv_report["commodity_signals"])}商品 + {len(deriv_report["term_structure"])}期限 + {len(deriv_report["index_futures_basis"])}基差 ({time.time()-t2:.2f}s)')

# Step 3: 体制检测
t3 = time.time()
regime_result = regime_engine.detect_regime()
print(f'[3] 体制检测: {regime_result["current_regime"]} (置信度={regime_result["confidence"]:.2%}) ({time.time()-t3:.2f}s)')

# Step 4: 状态分类
t4 = time.time()
classification_result = classifier.classify()
print(f'[4] 状态分类: {classification_result["market_state"]} (综合={classification_result["composite_score"]:.1f}) ({time.time()-t4:.2f}s)')

elapsed = time.time() - start
print(f'\n总耗时: {elapsed:.2f}s')
print(f'\n最终结论:')
print(f'  市场状态: {classification_result["market_state"]}')
print(f'  综合得分: {classification_result["composite_score"]:.1f}')
print(f'  三维: 估值={classification_result["valuation_score"]:.1f} 动量={classification_result["momentum_score"]:.1f} 体制={classification_result["regime_score"]:.1f}')
print(f'  置信度: {classification_result["confidence"]:.2%}')

  V7 完整管线集成测试

[1] 合约推导: 8商品 + 4股指 (0.00s)


ValueError: too many values to unpack (expected 3, got 8)

In [ ]:
# 清理资源
tdx.close()
db.close()
ak.close()
print('✅ 所有资源已释放')